# Week 03 - Lesson 03: Context Engineering

Context Engineering represents the strategic discipline of managing and optimizing the information available to AI agents throughout their execution lifecycle. As AI agents become increasingly sophisticated and handle complex, multi-step tasks, the ability to effectively manage context becomes critical for performance, cost optimization, and reliability.

<img src="https://media.licdn.com/dms/image/v2/D5612AQGDZChjFuwSGQ/article-cover_image-shrink_720_1280/B56ZgNc19TGUAY-/0/1752572335496?e=2147483647&v=beta&t=pnTU74zw6pZPFwAHOs4q1fSerQt9rEieth1jSS-3EOQ" alt="Context Engineering" width="700"/>

Context Engineering is the art and science of filling the context window with precisely the right information at each step of an agent's execution. Just as an operating system manages RAM to optimize CPU performance, context engineering manages the LLM's context window to optimize agent performance.

Understanding Context Engineering is crucial for:
- Building scalable AI agent systems
- Optimizing token usage and reducing operational costs
- Preventing context overflow and performance degradation
- Creating reliable, long-running agent workflows
- Implementing sophisticated multi-agent architectures

---

### Learning Objectives

1. Understand the fundamental principles of Context Engineering for AI agents
2. Master the four core strategies: Write, Select, Compress, and Isolate
3. Implement practical context management techniques for different agent architectures
4. Apply context engineering patterns to optimize agent performance and reduce costs

---

### The Context Engineering Challenge

Modern AI agents face a fundamental challenge: they must process vast amounts of information while operating within the constraints of limited context windows. This challenge becomes increasingly complex as agents:

- **Accumulate Information**: Each tool call, user interaction, and processing step adds to the context
- **Handle Long-Running Tasks**: Multi-step workflows can span hundreds of interactions
- **Process Diverse Data Types**: Code, documents, structured data, and conversation history
- **Coordinate Multiple Agents**: Information must be shared and synchronized across agent boundaries

---

### The Context Window as System Memory

Think of the LLM's context window as the system's RAM - it has limited capacity and must be managed strategically. Just as an operating system uses various techniques to manage memory efficiently, context engineering employs sophisticated strategies to optimize information flow.

---

### The Four Pillars of Context Engineering

*Context engineering strategies can be organized into four fundamental approaches:*

1. **Write Context** - Persisting Information Outside the Context Window
Saving important information to external storage systems that can be retrieved when needed.

2. **Select Context** - Retrieving Relevant Information
Intelligently pulling the right information into the context window at the right time.

3. **Compress Context** - Reducing Information Density
Condensing information while preserving essential details and decision-making capabilities.

4. **Isolate Context** - Separating Information Streams
Dividing context across different processing units or time periods to prevent interference.

<img src="https://blog.langchain.com/content/images/size/w1600/2025/07/image-4.png" alt="Context Engineering Diagram from Langchain" style="max-width:90%;">

Let's explore some of these strategies in detail with practical implementations.


---

## Environment Setup and Dependencies

Let's start by setting up our development environment with the necessary libraries for context engineering.


In [1]:
# Install required packages
%pip install --upgrade -q langchain langchain-openai langgraph chromadb python-dotenv langchain_chroma

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import uuid
from datetime import datetime
from dotenv import load_dotenv

# LangChain and LangGraph core imports
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

from langchain_core.tools import tool, StructuredTool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document

from langgraph.graph import (
    START, END, StateGraph, MessagesState
)
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph.message import add_messages

from typing import Annotated
from typing_extensions import TypedDict


In [19]:
# Set OpenAI API Key
load_dotenv()
os.environ['OPENAI_API_KEY'] = 'sk-proj-'

In [4]:
# Initialize OpenAI client
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1,
    api_key=os.getenv("OPENAI_API_KEY")
)

---

## 1. **Write Context**

The first pillar of context engineering involves saving important information to external storage systems. This allows agents to maintain access to critical data without consuming context window space. LLMs have a fixed context window, typically a few thousand to hundreds of thousands of tokens. Once the window is full, older information is lost unless it is explicitly summarized or stored elsewhere. Without persistence, agents:

* **Forget critical details** mid-task (e.g., a plan, a user preference, or intermediate reasoning steps).
* **Waste tokens** repeatedly re-feeding the same instructions or facts.
* **Lose continuity** across sessions, resulting in poor user experience.

By persisting information externally, developers **decouple memory from window size**, giving agents the ability to operate like humans who take notes, store reference material, and revisit past experiences.

<div></div>

---

### **1.1 Scratchpad Implementation**

A **scratchpad** acts as the agent’s *working memory*. It is a transient storage mechanism where the agent can “write notes to itself” during reasoning and execution. Later, those notes can be selectively pulled back into the context window.

> **Analogy:** Humans solving complex problems often jot down calculations or partial ideas on paper. Agents use scratchpads the same way — to keep track of sub-results, hypotheses, or plans without overloading the main prompt.

**Implementation Patterns:**

- **Append to agent state**

   The scratchpad can be maintained inside the agent’s state by simply appending to a variable during execution. This approach makes it easy to keep track of intermediate notes in a lightweight way.

- **Scratchpad as tools**

   The scratchpad can also be exposed as tools (`write_to_scratchpad`, `read_scratchpad`). Each call lets the agent record information into short-term memory, which can then be retrieved throughout execution.

In [5]:
class CustomState(MessagesState):
    scratchpad: str

scratchpad_storage = {"content": ""}

@tool
def write_to_scratchpad(note: str) -> str:
    """Write important information to the scratchpad memory. This will be added to existing content."""
    if scratchpad_storage["content"]:
        scratchpad_storage["content"] += f"\n{note}"
    else:
        scratchpad_storage["content"] = note
    return f"Added to scratchpad: {note}"

@tool
def read_scratchpad() -> str:
    """Read what's stored in the scratchpad."""
    content = scratchpad_storage["content"] if scratchpad_storage["content"] else "Empty"
    return f"Current scratchpad content:\n{content}"

@tool
def clear_scratchpad() -> str:
    """Clear all content from the scratchpad."""
    scratchpad_storage["content"] = ""
    return "Scratchpad cleared"

# Agent node that processes user input and updates scratchpad
def agent_node(state: CustomState):
    
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.1)
    
    # Create system message with scratchpad context
    system_prompt = f"""You are a helpful assistant with access to a scratchpad for storing important information.

    You can use the following tools to manage the scratchpad:
    - write_to_scratchpad: Write important user preferences or information to the scratchpad (this adds to existing content)
    - read_scratchpad: Read current scratchpad content  
    - clear_scratchpad: Clear the scratchpad

    During the conversation, you should write to the scratchpad whenever the user provides important information.
    If the user wants to know what you remember, you can use the read_scratchpad tool to retrieve the information.
    If the user wants to clear the scratchpad, you can use the clear_scratchpad tool to clear the scratchpad.
    When a user tells you about their preferences (like "I prefer brief answers" or "I like code examples"), you should write this to the scratchpad so it gets added to your memory.

    Always be helpful and use the scratchpad to remember user preferences and important details."""
    
    system_message = SystemMessage(content=system_prompt)
    
    # Bind tools to LLM
    llm_with_tools = llm.bind_tools([write_to_scratchpad, read_scratchpad, clear_scratchpad])
    messages = [system_message] + state["messages"]
    
    result = llm_with_tools.invoke(messages)
    
    return {"messages": [result]}

def should_continue(state: MessagesState):
    messages = state["messages"]
    last_message = messages[-1]
    if last_message.tool_calls:
        return "tools"
    return END

tool_node = ToolNode([write_to_scratchpad, read_scratchpad, clear_scratchpad])

builder = StateGraph(CustomState)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", should_continue, ["tools", END])
builder.add_edge("tools", "agent")

# Add checkpointer for memory
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

print("🤖 Simple Scratchpad Agent using StateGraph")
print("=" * 50)

test_inputs = [
    "Hello, how are you?",
    "I have preference to code examples only in JavaScript",
    "I prefer brief answers intead of long explanations",
    "Provide me what you have in your scratchpad",
    "Clear the scratchpad", 
    "What do you remember about me based only on the scratchpad?",
]

# Configuration with proper thread_id for the checkpointer
config = {"configurable": {"thread_id": "test_session"}}

for i, user_input in enumerate(test_inputs):
    print(f"\n👤 User: {user_input}")
        
    try:
        # Invoke the graph
        result = graph.invoke({"messages": [HumanMessage(content=user_input)]},config=config)
        print(result["messages"][-1].content)
        
    except Exception as e:
        print(f"Error: {e}")
    
    print("-" * 30)

🤖 Simple Scratchpad Agent using StateGraph

👤 User: Hello, how are you?
I'm here and ready to assist you! How can I help you today?
------------------------------

👤 User: I have preference to code examples only in JavaScript
Got it! I'll make sure to provide code examples in JavaScript for you. What else would you like to know or discuss?
------------------------------

👤 User: I prefer brief answers intead of long explanations
Understood! I'll keep my responses brief and to the point. Feel free to ask any questions you have. How can I assist you further?
------------------------------

👤 User: Provide me what you have in your scratchpad
Here's what I have in my scratchpad:
- User prefers code examples only in JavaScript
- User prefers brief answers over long explanations

Is there anything else you would like to add or ask about?
------------------------------

👤 User: Clear the scratchpad
The scratchpad has been cleared. Feel free to provide any new preferences or information you'd 

---

### **1.2 Long-term Memory Implementation**

While scratchpads provide short-term, task-bound persistence, they are limited to a single session. Long-term memory is what allows agents to learn, adapt, and personalize behavior across multiple interactions, effectively “remembering” the user or domain over time.

**Why Long-Term Memory Matters**

Continuity across sessions: Without persistence, every conversation starts from zero, forcing the user to restate preferences and facts.

- Personalization: Agents can adapt responses based on stored preferences, style, and history.
- Knowledge accumulation: Over time, the agent builds a knowledge base of facts, procedures, and outcomes.
- Efficiency: Reduces token usage by avoiding repeated re-ingestion of static or known facts.
- Reflection and synthesis: Enables higher-level “learning” where past interactions are distilled into reusable insights or procedures.

[Pinecone’s guide](https://www.pinecone.io/learn/context-engineering/) to context engineering emphasizes that “writing” memories outside of the LLM’s window is the only way to overcome the context window bottleneck. This shift transforms agents from stateless query processors into adaptive systems.

**Design Principles**

- **Persistence**: Store memories in an external, durable database (e.g., vector DB). This ensures that user preferences, facts, and procedures survive across sessions.

- **Semantic retrieval**: Index memories as embeddings, enabling the agent to recall relevant entries with similarity search rather than scanning entire logs.

- **Typed memories**: Categorize memories as:

<img src="https://blog.langchain.com/content/images/size/w1600/2025/07/image-6.png" alt="Long-term memory architecture diagram" style="max-width: 80%;">

**Implementation Example**

In the example below, we will implement a **Write Context** technique with **long-term memory using vector storage**, while also demonstrating **Select Context** concepts through intelligent semantic retrieval. This implementation combines both Context Engineering strategies in an integrated system that allows for both persisting information and retrieving it in a contextually relevant manner.

---

## 2. **Select Context**

The second pillar focuses on intelligently selecting and retrieving the most relevant information for the current task. This involves sophisticated retrieval mechanisms that understand context and relevance.

---

### **2.1 Context-Aware Retrieval System**

**In this section, we will implement the `Write Context` technique from `1.2 Long-term Memory`, and based on that stored memory, we will learn how to `Select Context` by intelligently retrieving only the most relevant information.**

The challenge of memory selection becomes critical when building personalized AI agents. While simple agents might use fixed files or rules (like *Claude Code's* `CLAUDE.md` or *Cursor's rules files*), sophisticated agents need to intelligently select from a growing collection of user memories, preferences, and contextual information.

The key is implementing a retrieval system that can understand what memories are truly relevant for each specific task. This goes beyond simple keyword matching - it requires semantic understanding of both the user's current request and the stored memories. 

For example, when a user asks about deployment, the system should retrieve deployment-related memories, user preferences about deployment tools, and any previous deployment experiences, while filtering out irrelevant personal information or unrelated technical details.

This selective retrieval is essential for creating a personalized experience where the agent feels like it truly "knows" the user, without overwhelming the context window with irrelevant information or accidentally injecting inappropriate memories into responses.


In [6]:
# Global memory storage
memory_metadata = {}

# Initialize vector store
embeddings = OpenAIEmbeddings()
vectorstore = Chroma(
    persist_directory="./memory_db",
    embedding_function=embeddings
)

# Define the state
class MemoryState(MessagesState):
    pass

#### What This Code Does

This implementation demonstrates both Write Context and Select Context strategies from Context Engineering:

**Write Context (Memory Storage)**

- `store_memory()` tool: Persists user information outside the LLM's context window
- Vector Database: Uses ChromaDB to store memories with semantic embeddings
- Metadata System: Categorizes memories by type (user_preference, fact, procedure, general)

**Select Context (Intelligent Retrieval)**

- `retrieve_memories()` tool: Uses semantic search to find relevant memories
- Context-Aware Selection: Filters memories based on current query and memory type
- Similarity Scoring: Ranks retrieved memories by relevance to the current task
- `get_memory_by_type()` tool: Allows targeted retrieval of specific memory categories

In [7]:
# Memory Tools
@tool
def store_memory(content: str, memory_type: str = "general", 
                importance: int = 5, tags: str = "") -> str:
    """
    Store a memory with semantic search capabilities.
    
    Args:
        content: The memory content to store
        memory_type: Type of memory (user_preference, fact, procedure, etc.)
        importance: Importance level (1-10)
        tags: Comma-separated tags for categorization
    """    
    memory_id = f"memory_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    # Create metadata for the memory
    metadata = {
        "memory_type": memory_type,
        "importance": importance,
        "timestamp": datetime.now().isoformat(),
        "tags": tags
    }

    # Add memory to vector store
    vectorstore.add_texts(
        texts=[content],
        metadatas=[metadata],
        ids=[memory_id]
    )
    
    memory_metadata[memory_id] = metadata
    print(f"🧠 Memory stored: {memory_id} (type: {memory_type})")
    return f"✅ Stored memory: {content[:50]}..."

@tool
def retrieve_memories(query: str, memory_type: str = None, 
                     limit: int = 5) -> str:
    """
    Retrieve relevant memories based on semantic similarity.
    
    Args:
        query: Search query
        memory_type: Filter by memory type (optional)
        limit: Maximum number of memories to return
    """
    # Build filter for memory type if specified
    filter_dict = None
    if memory_type:
        filter_dict = {"memory_type": memory_type}
    
    # Perform semantic search
    results = vectorstore.similarity_search_with_score(
        query, k=limit, filter=filter_dict
    )
    
    memories = []
    for doc, score in results:
        memory_id = doc.metadata.get('_id', 'unknown')
        memories.append({
            "content": doc.page_content,
            "score": score,
            "metadata": memory_metadata.get(memory_id, {}),
            "id": memory_id
        })
    
    print(f"🔍 Retrieved {len(memories)} memories for query: '{query}'")
    
    if not memories:
        return "No memories found for that query."
    
    # Format results for display
    result_text = f"Found {len(memories)} memories:\n"
    for i, mem in enumerate(memories, 1):
        result_text += f"{i}. {mem['content']} (score: {mem['score']:.3f})\n"
    
    return result_text

@tool
def get_memory_by_type(memory_type: str) -> str:
    """Get all memories of a specific type."""

    filter_dict = {"memory_type": memory_type}
    results = vectorstore.get(where=filter_dict)
    
    if not results['documents']:
        return f"No memories found for type: {memory_type}"
    
    memories = []
    for i, content in enumerate(results['documents']):
        memory_id = results['ids'][i]
        memories.append({
            "content": content,
            "metadata": memory_metadata.get(memory_id, {}),
            "id": memory_id
        })
    
    result_text = f"Found {len(memories)} memories of type '{memory_type}':\n"
    for i, mem in enumerate(memories, 1):
        result_text += f"{i}. {mem['content']}\n"
    
    return result_text

In [8]:
# Agent node for memory management
def memory_agent_node(state: MemoryState):
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.1)
    
    system_prompt = """You are a helpful memory management assistant with access to long-term memory storage.

    You can:
    1. Store memories using store_memory(content, memory_type, importance, tags)
    2. Retrieve memories using retrieve_memories(query, memory_type, limit)
    3. Get memories by type using get_memory_by_type(memory_type)

    Memory types include only: user_preference, fact, procedure, general
    Importance levels: 1-10 (10 being most important)
    Tags should be comma-separated strings

    Store in memory whenever you encounter information that is important to the user.  

    When users ask you to remember something, store it as a memory.
    When users ask what you remember, use retrieve_memories to find relevant information.
    Always be helpful and use the memory system to maintain context across conversations."""
    
    system_message = SystemMessage(content=system_prompt)
    
    # Bind tools to LLM
    llm_with_tools = llm.bind_tools([store_memory, retrieve_memories, get_memory_by_type])
    
    # Create messages list with system message and conversation history
    messages = [system_message] + state["messages"]
    
    result = llm_with_tools.invoke(messages)
    
    return {"messages": [result]}

# Helper function to check if we need to call tools
def should_continue_memory(state: MemoryState):
    last_message = state["messages"][-1]
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        return "tools"
    return "end"

# Build the memory agent graph
memory_builder = StateGraph(MemoryState)
memory_builder.add_node("agent", memory_agent_node)

# Use the ToolNode component for memory tools
memory_tools = [store_memory, retrieve_memories, get_memory_by_type]
memory_tool_node = ToolNode(memory_tools)
memory_builder.add_node("tools", memory_tool_node)

memory_builder.add_edge(START, "agent")
memory_builder.add_conditional_edges("agent", should_continue_memory)
memory_builder.add_edge("tools", "agent")

# Compile the memory agent
memory_checkpointer = InMemorySaver()
memory_agent = memory_builder.compile(checkpointer=memory_checkpointer)

# Configuration with proper thread_id for the checkpointer
memory_config = {"configurable": {"thread_id": "memory_session"}}

In [9]:
user_input = "Remember that I prefer detailed technical explanations with code examples"

result = memory_agent.invoke({"messages": [("user", user_input)]}, config=memory_config)

for message in result["messages"]:
    message.pretty_print()

🧠 Memory stored: memory_20250923_170828 (type: user_preference)


Task agent with path ('__pregel_pull', 'agent') wrote to unknown channel branch:to:end, ignoring it.


================================ Human Message =================================

Remember that I prefer detailed technical explanations with code examples
================================== Ai Message ==================================
Tool Calls:
  store_memory (call_CsbyLqlJu3HAYAuC0JI6zJSf)
 Call ID: call_CsbyLqlJu3HAYAuC0JI6zJSf
  Args:
    content: User prefers detailed technical explanations with code examples
    memory_type: user_preference
    importance: 8
    tags: technical,code
================================= Tool Message =================================
Name: store_memory

✅ Stored memory: User prefers detailed technical explanations with ...
================================== Ai Message ==================================

I've noted that you prefer detailed technical explanations with code examples. If you have any specific preferences or requirements regarding this, feel free to let me know!


In [10]:
user_input = "Store the fact that our company's primary database is PostgreSQL running on AWS RDS"

result = memory_agent.invoke({"messages": [("user", user_input)]}, config=memory_config)

for message in result["messages"][-4:]:
    message.pretty_print()

🧠 Memory stored: memory_20250923_170830 (type: fact)


Task agent with path ('__pregel_pull', 'agent') wrote to unknown channel branch:to:end, ignoring it.


================================ Human Message =================================

Store the fact that our company's primary database is PostgreSQL running on AWS RDS
================================== Ai Message ==================================
Tool Calls:
  store_memory (call_ENVAm32SBRI5KF8u3ykKRKYZ)
 Call ID: call_ENVAm32SBRI5KF8u3ykKRKYZ
  Args:
    content: Company's primary database is PostgreSQL running on AWS RDS
    memory_type: fact
    importance: 7
    tags: database,PostgreSQL,AWS,RDS
================================= Tool Message =================================
Name: store_memory

✅ Stored memory: Company's primary database is PostgreSQL running o...
================================== Ai Message ==================================

The fact that our company's primary database is PostgreSQL running on AWS RDS has been stored. If you need this information in the future or have any related queries, feel free to ask!


In [11]:
user_input = "What do you remember about my general preferences?"

result = memory_agent.invoke({"messages": [("user", user_input)]}, config=memory_config)

for message in result["messages"][-4:]:
    message.pretty_print()

Task agent with path ('__pregel_pull', 'agent') wrote to unknown channel branch:to:end, ignoring it.


================================ Human Message =================================

What do you remember about my general preferences?
================================== Ai Message ==================================
Tool Calls:
  get_memory_by_type (call_EB5pHq9iYXYJPwk2TWEZfFJ2)
 Call ID: call_EB5pHq9iYXYJPwk2TWEZfFJ2
  Args:
    memory_type: user_preference
================================= Tool Message =================================
Name: get_memory_by_type

Found 2 memories of type 'user_preference':
1. User prefers detailed technical explanations with code examples
2. User prefers detailed technical explanations with code examples

================================== Ai Message ==================================

I remember that your general preference is for detailed technical explanations with code examples. If there are any other preferences you would like me to remember or update, feel free to share!


In [12]:
user_input = "Show me all stored facts"

result = memory_agent.invoke({"messages": [("user", user_input)]}, config=memory_config)

for message in result["messages"][-4:]:
    message.pretty_print()

Task agent with path ('__pregel_pull', 'agent') wrote to unknown channel branch:to:end, ignoring it.


================================ Human Message =================================

Show me all stored facts
================================== Ai Message ==================================
Tool Calls:
  get_memory_by_type (call_CV3CaU01OMjeAKXWyXLZcWSz)
 Call ID: call_CV3CaU01OMjeAKXWyXLZcWSz
  Args:
    memory_type: fact
================================= Tool Message =================================
Name: get_memory_by_type

Found 2 memories of type 'fact':
1. Company's primary database is PostgreSQL running on AWS RDS
2. Company's primary database is PostgreSQL running on AWS RDS

================================== Ai Message ==================================

Here are all the stored facts:
1. Company's primary database is PostgreSQL running on AWS RDS

If you need more details or have any other specific facts you want to store or retrieve, feel free to ask!


In [13]:
user_input = "What do you remember about my user preferences only?"

result = memory_agent.invoke({"messages": [("user", user_input)]}, config=memory_config)

for message in result["messages"][-4:]:
    message.pretty_print()

Task agent with path ('__pregel_pull', 'agent') wrote to unknown channel branch:to:end, ignoring it.


================================ Human Message =================================

What do you remember about my user preferences only?
================================== Ai Message ==================================
Tool Calls:
  get_memory_by_type (call_0ZGiTDvoz3nDmhYrFTWgaaVz)
 Call ID: call_0ZGiTDvoz3nDmhYrFTWgaaVz
  Args:
    memory_type: user_preference
================================= Tool Message =================================
Name: get_memory_by_type

Found 2 memories of type 'user_preference':
1. User prefers detailed technical explanations with code examples
2. User prefers detailed technical explanations with code examples

================================== Ai Message ==================================

I remember the following user preferences:
1. User prefers detailed technical explanations with code examples

If you have any new preferences or updates to your existing preferences, feel free to share them with me!


---

### **2.2 Tool Selection and Filtering**

Effective tool selection ensures that agents only access the most relevant tools, minimizing confusion and improving overall performance.

#### Dynamic Tool Selection with Semantic Search

When agents are exposed to a large number of tools, performance can degrade. This often happens because tool descriptions overlap, leading the model to become uncertain about which tool to invoke. To address this, we can apply **retrieval-augmented generation (RAG)** over tool descriptions, fetching only the most relevant tools for each task. Research has shown that this approach can improve tool selection accuracy by up to threefold.

A practical method is to use **vector search** to dynamically rank and select tools based on the conversation context. This ensures that the agent’s toolset remains both targeted and adaptive, even in environments with hundreds or thousands of available tools. This approach aligns with strategies outlined in the [LangGraph documentation on handling large tool collections](https://langchain-ai.github.io/langgraph/how-tos/many-tools/).

**Key Benefits:**

* **Context-aware**: Selects tools based on the current conversation context
* **Scalable**: Supports hundreds or thousands of tools without overwhelming the agent
* **Efficient**: Reduces token usage by limiting tool exposure
* **Accurate**: RAG-driven filtering mitigates confusion from overlapping tool descriptions, with measurable improvements in accuracy
* **Adaptive**: Can re-select tools as the conversation evolves


In [14]:
def create_business_tool(tool_name: str) -> StructuredTool:
    """Create a placeholder tool with realistic description."""
    descriptions = {
        "customer_database_query": "Query customer information from the database",
        "inventory_check": "Check product availability and stock levels",
        "sales_report_generator": "Generate sales reports and analytics",
        "email_campaign_manager": "Create and manage email marketing campaigns",
        "financial_analytics": "Analyze financial data and generate insights",
        "user_authentication": "Handle user login and authentication",
        "payment_processor": "Process payments and handle transactions",
        "shipping_tracker": "Track package delivery and shipping status",
        "product_catalog_search": "Search and browse product catalog",
        "support_ticket_system": "Manage customer support tickets"
    }
    
    def tool_function(query: str = "") -> str:
        return f"Executing {tool_name} with query: {query}"
    
    return StructuredTool.from_function(
        tool_function,
        name=tool_name,
        description=descriptions.get(tool_name, f"Tool for {tool_name}")
    )

    # Business tools for demonstration
business_tools = [
    "customer_database_query",
    "inventory_check", 
    "sales_report_generator",
    "email_campaign_manager",
    "financial_analytics",
    "user_authentication",
    "payment_processor",
    "shipping_tracker",
    "product_catalog_search",
    "support_ticket_system"
]

# Create a tool registry with unique IDs
tool_registry = {
    str(uuid.uuid4()): create_business_tool(tool_name) for tool_name in business_tools
}

# Create vector store for tool selection
embeddings = OpenAIEmbeddings()
vector_store = InMemoryVectorStore(embedding=embeddings)

# Create documents for each tool
tool_documents = [
    Document(
        page_content=tool.description,
        id=tool_id,
        metadata={"tool_name": tool.name},
    )
    for tool_id, tool in tool_registry.items()
]

# Add documents to vector store
document_ids = vector_store.add_documents(tool_documents)

# Initialize LLM
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.1)

In [15]:
# State
class State(TypedDict):
    messages: Annotated[list, add_messages]
    selected_tools: list[str]

builder = StateGraph(State)

# Retrieve all available tools from the tool registry
tools = list(tool_registry.values())
llm = ChatOpenAI()

# The agent processes the current state by binding selected tools to the LLM.
def agent(state: State):
    # Select the tools based on the state's selected_tools list.
    selected_tools = [tool_registry[id] for id in state["selected_tools"]]
    # Bind the selected tools to the LLM for the current interaction.
    llm_with_tools = llm.bind_tools(selected_tools)
    # Invoke the LLM with the current messages and return the updated message list.
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


# Select the tools based on the user's last message content.
def select_tools(state: State):
    last_user_message = state["messages"][-1]
    query = last_user_message.content
    tool_documents = vector_store.similarity_search(query)
    return {"selected_tools": [document.id for document in tool_documents]}

# Create the graph
builder.add_node("agent", agent)
builder.add_node("select_tools", select_tools)

tool_node = ToolNode(tools=tools)
builder.add_node("tools", tool_node)

builder.add_conditional_edges("agent", tools_condition, path_map=["tools", "__end__"])
builder.add_edge("tools", "agent")
builder.add_edge("select_tools", "agent")
builder.add_edge(START, "select_tools")
graph = builder.compile()

In [16]:
user_input = "Generate a sales report for this month"

result = graph.invoke({"messages": [("user", user_input)]})
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Generate a sales report for this month
================================== Ai Message ==================================
Tool Calls:
  sales_report_generator (call_TNYJ4ZLg1gTFMYuxQfCXwOPj)
 Call ID: call_TNYJ4ZLg1gTFMYuxQfCXwOPj
  Args:
================================= Tool Message =================================
Name: sales_report_generator

Executing sales_report_generator with query: 
================================== Ai Message ==================================

I have generated the sales report for this month. Is there anything else you would like to do?


In [17]:
user_input = "Track a package delivery"

result = graph.invoke({"messages": [("user", user_input)]})
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Track a package delivery
================================== Ai Message ==================================
Tool Calls:
  shipping_tracker (call_H6jSpKVHQBtqIghGozlat14u)
 Call ID: call_H6jSpKVHQBtqIghGozlat14u
  Args:
================================= Tool Message =================================
Name: shipping_tracker

Executing shipping_tracker with query: 
================================== Ai Message ==================================

I am currently tracking the package delivery for you. I will provide you with the status once I have the information.


In [18]:
user_input = "Process a payment of $99.99"

result = graph.invoke({"messages": [("user", user_input)]})
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Process a payment of $99.99
================================== Ai Message ==================================
Tool Calls:
  payment_processor (call_2ToCPrqgsXyIpjWhWi6UaYwx)
 Call ID: call_2ToCPrqgsXyIpjWhWi6UaYwx
  Args:
    query: $99.99
================================= Tool Message =================================
Name: payment_processor

Executing payment_processor with query: $99.99
================================== Ai Message ==================================

The payment of $99.99 has been processed successfully.
